# COMPSCI5094 Coursework - Conversational Interfaces: Premier League Dialogue Agent
**Author**: Patricia Natasha Munginga

**Student ID**: 3033388M

**Date**: February 2026

**Project**: Task‑Oriented Dialogue System for Premier League Information

This notebook documents the development of a task‑oriented conversational agent that provides information about English Premier League teams and personnel using a football data API and an LLM‑based NLU component.
The focus is on defining intents and slots, implementing robust intent detection and slot filling with a local LLM, integrating real‑time football data via an external API and demonstrating end‑to‑end dialogues that satisfy user queries while enabling clear evaluation of NLU performance.

### Setup and Imports

In [7]:
"""Installs and Imports"""

# Install the Ollama python library
%pip install ollama

# Import the Ollama python library
import ollama

# API functionality
import requests
import json

Note: you may need to restart the kernel to use updated packages.


### Configuration

In [8]:
# Create a client connected to the local Ollama server
client = ollama.Client(host= 'http://localhost:11434') ##makatea.dcs.gla.ac.uk:11434, go to uni gf glasgow vpn and set it up

# Ensure the required model is available locally (downloads it if needed)
!ollama pull qwen3:4b-instruct
!ollama pull phi3:mini

# Define the model name to use throughout the notebook
MODEL = 'qwen3:4b-instruct'  
#MODEL = "phi3:mini"

# Database and API 
FOOTBALL_DATA_API_KEY = "7b428478107f4bdb98af277f262c4350"
FOOTBALL_DATA_BASE_URL = "https://api.football-data.org/v4"
PREMIER_LEAGUE_CODE = "PL" # English Premier League

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 85e4a5b7b8ef: 100% ▕██████████████████▏ 2.5 GB                         
pulling eade0a07cac7: 100% ▕██████████████████▏ 1.4 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling 0914c7781e00: 100% ▕██████████████████▏  119 B                         
pulling b72accf9724e: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 633fc5be925f: 100% ▕██████████████████▏ 2.2 GB                         
pulling fa8235e5b48f: 100% ▕██████████████████▏ 1.1 KB                         
pulling 542b217f179c: 100% ▕██████████████████▏  148 B                         
pullin

### API Functions

In [9]:
def call_football_data_api(endpoint, params=None, verbose=True):
    """
    Generic helper function to call the football-data API.

    endpoint: string like 'teams/57'
    params: dictionary of query parameters (e.g., {'status': 'SCHEDULED'})
    verbose: if True, prints basic debug info
    """
    headers = {'X-Auth-Token': FOOTBALL_DATA_API_KEY}
    url = f"{FOOTBALL_DATA_BASE_URL}/{endpoint}"

    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)

        if verbose:
            print(f"[API] GET {url} params={params} status={response.status_code}")

        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        # Return a structured error (better for tool outputs)
        return {"error": str(e), "endpoint": endpoint, "params": params}

def filter_premier_league_matches(matches):
    """
    Premier League-only filter

    Purpose: /teams/{id}/matches may include PL + other competitions (e.g., CL).
    This system is Premier League only
    """
    return [m for m in matches if m.get('competition', {}).get('code') == PREMIER_LEAGUE_CODE]    
      
def get_team_id_by_name(team_name):
    """
    Helper function to get team ID from team name

    SLOT: team (required)
    Endpoint used: /v4/competitions/PL/teams
    Purpose: Resolve team name to team ID
    """
    result = call_football_data_api(f'competitions/{PREMIER_LEAGUE_CODE}/teams', verbose = False)
    
    if not result or 'error' in result:
        return None
    
    teams = result.get('teams', [])
    team_name_lower = team_name.lower().strip()
    
    # 1) Exact match on full or short name
    for team in teams:
        if team['name'].lower() == team_name_lower or \
           team['shortName'].lower() == team_name_lower:
            return team['id']
    
    # 2) Exact match on TLA (3-letter code)
    for team in teams:
        if team['tla'].lower() == team_name_lower:
            return team['id']
    
    # 3) Partial match fallback
    for team in teams:
        if team_name_lower in team['name'].lower():
            return team['id']
    
    return None 

def get_manager_info(team_name):
    """
    Get information about a team's manager/coach

    SLOT: manager
    Endpoint used: /v4/teams/{id}
    Purpose: Returns detailed team info including coach/manager field
    """
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}
    
    result = call_football_data_api(f'teams/{team_id}', verbose = False)

    if not result or 'error' in result:
        return result if result else {"error": "API call failed"}
    
    coach = result.get('coach')

    if not coach:
        return {"error": "Manager information not available"}
    
    return {
        'manager': coach.get('name', 'N/A'),
        'nationality': coach.get('nationality', 'N/A'),
        'team': result.get('name', 'N/A')
    }

def get_standings_info(team_name):
    """
    Get standings for a team

    SLOTS:
      - leaguePosition
      - numGamesPlayed
      - winLossRecord

    Endpoint used: /v4/competitions/PL/standings
    Purpose: Returns league table including position, games played, wins, draws, losses
    """

    # Step 1: Resolve team name - team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Call Premier League standings endpoint
    result = call_football_data_api(
        f'competitions/{PREMIER_LEAGUE_CODE}/standings',
        verbose = False
    )

    if not result or 'standings' not in result:
        return {"error": "Standings data not available"}

    # Step 3: Find the TOTAL standings table
    for standing_type in result['standings']:
        if standing_type['type'] == 'TOTAL':

            # Step 4: Find this team in the league table
            for team_row in standing_type['table']:
                if team_row['team']['id'] == team_id:

                    return {
                        "team": team_row['team']['name'],
                        "leaguePosition": team_row['position'],
                        "numGamesPlayed": team_row['playedGames'],
                        "winLossRecord": f"{team_row['won']}-{team_row['draw']}-{team_row['lost']}"
                    }

    return {"error": "Team not found in standings"}

def get_next_match(team_name):
    """
    Get next match for a team

    SLOTS:
       - nextGameDate
       - nextOpponent

    Endpoint used: /v4/teams/{id}/matches?status=SCHEDULED
    Purpose: Returns upcoming matches
    """

    # Step 1: Resolve team name - team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Get scheduled matches
    result = call_football_data_api(
        f'teams/{team_id}/matches',
        {'status': 'SCHEDULED'},
        verbose = False
    )

    if not result or 'matches' not in result:
        return {"error": "Match data not available"}

    matches = result['matches']

    # Step 3: Filter Premier League matches only
    pl_matches = filter_premier_league_matches(matches)

    if not pl_matches:
        return {"error": "No upcoming Premier League matches found"}

    # Step 4: Sort by utcDate ascending → next match first
    pl_matches = sorted(pl_matches, key=lambda m: m['utcDate'])
    next_match = pl_matches[0]

    # Step 5: Determine opponent
    if next_match['homeTeam']['id'] == team_id:
        opponent = next_match['awayTeam']['name']
    else:
        opponent = next_match['homeTeam']['name']

    return {
        "team": team_name,
        "nextGameDate": next_match['utcDate'],
        "nextOpponent": opponent
    }


def get_last_match(team_name):
    """
    Get last played match for a team

    SLOTS:
     - lastOpponent
     - lastScore

    Endpoint used: /v4/teams/{id}/matches?status=FINISHED
    Purpose: Returns completed matches including full-time score
    """

    # Step 1: Resolve team name → team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Get finished matches
    result = call_football_data_api(
        f"teams/{team_id}/matches",
        {"status": "FINISHED"},
        verbose = False
    )

    if not result or "matches" not in result:
        return {"error": "Match data not available"}

    matches = result["matches"]

    # Step 3: Filter Premier League matches only
    pl_matches = filter_premier_league_matches(matches)

    if not pl_matches:
        return {"error": "No finished Premier League matches found"}

    # Step 4: Sort by utcDate descending → most recent finished match first
    pl_matches = sorted(pl_matches, key=lambda m: m["utcDate"], reverse=True)
    last_match = pl_matches[0]

    # Step 5: Determine opponent
    if last_match["homeTeam"]["id"] == team_id:
        opponent = last_match["awayTeam"]["name"]
    else:
        opponent = last_match["homeTeam"]["name"]

    # Step 6: Extract score
    full_time = last_match["score"]["fullTime"]
    last_score = f"{full_time['home']} - {full_time['away']}"

    return {
        "team": team_name,
        "lastOpponent": opponent,
        "lastScore": last_score
    }


def get_playing_now(team_name):
    """
    Check if a team is currently playing a match
    
    SLOT: playingNow

    Endpoint used: /v4/teams/{id}/matches?status=IN_PLAY
    Purpose: Checks if team currently has a live match
    """

    # Step 1: Resolve team name - team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Get live matches
    result = call_football_data_api(
        f"teams/{team_id}/matches",
        {"status": "IN_PLAY"},
        verbose = False
    )

    if not result or "matches" not in result:
        return {"playingNow": False}

    matches = result["matches"]

    # Step 3: Filter Premier League matches only and Check if any PL match is live
    if not result or 'error' in result or "matches" not in result:
        return {"team": team_name, "playingNow": False}

    pl_matches = filter_premier_league_matches(result["matches"])

    return {
        "team": team_name,
        "playingNow": len(pl_matches) > 0
    }

### Dialogue Manager

In [10]:
"""
DIALOGUE MANAGER

Responsible for:
1. Receiving tool calls from the LLM (intent + slot extraction).
2. Executing the corresponding backend function.
3. Returning structured API results back to the LLM.
4. Generating the final natural language response.

This separates:
- NLU (model selects tool)
- Backend execution (Python functions + API)
- NLG (model generates final response)

Ensures deterministic, tool-grounded behaviour.
"""
def execute_tool(tool_call):
    """
    Execute the appropriate backend function based on the
    LLM-selected tool name.
    """

    function_name = tool_call.function['name']
    arguments = tool_call.function['arguments']

    if function_name == 'get_manager_info':
        return get_manager_info(arguments['team_name'])

    elif function_name == 'get_standings_info':
        return get_standings_info(arguments['team_name'])

    elif function_name == 'get_next_match':
        return get_next_match(arguments['team_name'])

    elif function_name == 'get_last_match':
        return get_last_match(arguments['team_name'])

    elif function_name == 'get_playing_now':
        return get_playing_now(arguments['team_name'])

    else:
        return {"error": f"Unknown function: {function_name}"}


### Main Loop

In [11]:
def run_football_agent():

    print("ASSISTANT: Hello, how can I help you?\n")

    MAX_TURNS = 20
    turn_count = 0
    last_team = None

    while turn_count < MAX_TURNS:

        user_message = input("USER: ").strip()

        if not user_message:
            continue

        print(f"USER: {user_message}\n")

        # NLU STEP (Intent + Slots as JSON) 

        nlu_prompt = [
            {
                "role": "system",
                "content": """
        You are a Premier League football assistant.

        Extract structured information from the user request.

        Return ONLY valid JSON:

        {
        "intent": "GetInfo or Other",
        "function": "get_manager_info | get_standings_info | get_next_match | get_last_match | get_playing_now | null",
        "team_name": "string or null"
        }

        IMPORTANT RULES:

        - If the user asks for football information → intent = "GetInfo"
        - If the user says goodbye, thanks, okay, that's all, or any closing statement → intent = "Other"
        - If the message is not asking for football info → intent = "Other"
        - When intent = "Other", function and team_name MUST be null.
        """
            },
            {"role": "user", "content": user_message}
        ]

        response = client.chat(
            model=MODEL,
            messages=nlu_prompt,
            options={"temperature": 0}
        )

        import json

        try:
            parsed = json.loads(response["message"]["content"])
        except:
            print("ASSISTANT: Sorry, I didn't understand.\n")
            continue

        intent = parsed["intent"]
        function_name = parsed["function"]
        team_name = parsed["team_name"]

        # Safety rule: if no function detected, treat as Other
        if intent == "GetInfo" and not function_name:
            intent = "Other"

        # Pronoun memory support
        if team_name:
            last_team = team_name
        else:
            team_name = last_team

        detected_slots = {"team": team_name, "info": function_name}

        print(f"- Intent: {intent}")
        print(f"- Slots: {detected_slots}\n")

        # Exit condition
        if intent == "Other":
            print("ASSISTANT: Bye.\n")
            break

        if not team_name:
            print("ASSISTANT: Which team are you asking about?\n")
            continue

        # ROUTING (same logic as tools) 

        if function_name == "get_manager_info":
            result = get_manager_info(team_name)

        elif function_name == "get_standings_info":
            result = get_standings_info(team_name)

        elif function_name == "get_next_match":
            result = get_next_match(team_name)

        elif function_name == "get_last_match":
            result = get_last_match(team_name)

        elif function_name == "get_playing_now":
            result = get_playing_now(team_name)

        else:
            print("ASSISTANT: Sorry, I can't help with that.\n")
            continue

        # NLG STEP

        nlg_prompt = [
            {
                "role": "system",
                "content": "Explain the football data clearly and naturally."
            },
            {
                "role": "user",
                "content": f"User asked: {user_message}\n\nData:\n{result}"
            }
        ]

        final_response = client.chat(
            model=MODEL,
            messages=nlg_prompt,
            options={"temperature": 0.6}
        )

        assistant_message = final_response["message"]["content"]

        print(f"ASSISTANT: {assistant_message}\n")

        turn_count += 1

if __name__ == "__main__":
    run_football_agent()        

ASSISTANT: Hello, how can I help you?

USER: Hello, I need to know the record of Manchester City?

- Intent: GetInfo
- Slots: {'team': 'Manchester City', 'info': 'get_standings_info'}

ASSISTANT: Hello! Here's the record for Manchester City FC based on the data:

- **League Position**: 2nd (which means they are in second place in the league)
- **Total Games Played**: 26
- **Win-Loss-Draw Record**: 16 wins, 5 losses, and 5 draws

So, Manchester City has a strong record with 16 wins, showing they've performed well in their matches so far. They are currently in a very competitive position in the league, just behind the top team. Let me know if you'd like more details! 😊⚽

USER: Are they playing right now?

- Intent: GetInfo
- Slots: {'team': 'Manchester City', 'info': 'get_playing_now'}

ASSISTANT: No, Manchester City is not playing right now. The data shows that their match is not ongoing.

USER: Who did they play last?

- Intent: GetInfo
- Slots: {'team': 'Manchester City', 'info': 'get